# Sentiment Comparison: FinBERT vs RoBERTa vs FinBERT-Tone

This notebook compares three transformer models on a large dataset (~7M rows) of financial market news:

- **FinBERT** ([`ProsusAI/finbert`](https://huggingface.co/ProsusAI/finbert)) — BERT fine-tuned on the Financial PhraseBank.
- **RoBERTa** ([`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)) — a general-purpose (non-financial) sentiment RoBERTa, used here as a baseline contrast to the finance-tuned models.
- **FinBERT-Tone** ([`yiyanghkust/finbert-tone`](https://huggingface.co/yiyanghkust/finbert-tone)) — BERT fine-tuned on analyst reports for financial tone classification.

All three models predict `positive` / `negative` / `neutral`, so their outputs are directly comparable.

**What this notebook does:**
1. Downloads your dataset from Google Drive and mounts your Drive for outputs.
2. Loads the data and auto-detects the text column (override if needed); optionally takes a random sample via `SAMPLE_N`.
3. Runs all three models in resumable chunks — each finished chunk is checkpointed to Drive as a Parquet shard, so a Colab disconnect costs at most one chunk of work.
4. Combines the shards and saves the final results (original data + each model's label and per-class probabilities) as Parquet on your Drive.
5. Produces a comparison: label distributions, pairwise agreement, and disagreement examples.

> Run this in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Rough cost: ~2 GPU-hours per million rows for all three models, so validate on a sample first (`SAMPLE_N` in the config cell).

## 1. Install dependencies

In [ ]:
%pip install -q gdown transformers torch pandas pyarrow tqdm scikit-learn matplotlib


## 2. Configuration

Edit the values in this cell to match your dataset.

In [ ]:
import os

# --- Google Drive source -----------------------------------------------
GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1q_mca4_Uk3oHkHHMnCtyBL8QWZirR05c"
DATA_DIR = "data"  # local folder the Drive folder will be downloaded into

# --- Dataset options -----------------------------------------------------
# Path to a specific file inside DATA_DIR. Leave as None to auto-pick the
# first CSV/XLSX found in DATA_DIR.
DATA_FILE = None

# Name of the column containing the news text to classify.
# Leave as "auto" to auto-detect from a list of common column names.
TEXT_COLUMN = "auto"

# Optional: name of an existing ground-truth sentiment column, if your
# dataset has one. Leave as "auto" to auto-detect, or None if there is none.
LABEL_COLUMN = "auto"

# Random sample size to process. The full dataset is ~7M rows, which takes
# on the order of 15 GPU-hours for all three models — start with a sample
# to validate everything, then set SAMPLE_N = None to process all rows.
SAMPLE_N = 100_000
RANDOM_SEED = 42

# --- Output ---------------------------------------------------------------
# If True (and running in Colab), mount Google Drive and store checkpoints
# and final results there so they survive runtime disconnects.
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/finsent_results"
OUTPUT_BASENAME = "sentiment_comparison_results"

# --- Inference options ------------------------------------------------
BATCH_SIZE = 64        # raise to 128 on an A100, lower to 16 on CPU
MAX_LENGTH = 128       # tokens; plenty for headlines, keeps inference fast
CHUNK_SIZE = 25_000    # rows per checkpoint shard


## 2b. Storage setup

With ~7M rows, everything important (checkpoints and final results) should live on **Google Drive**, not the Colab machine's local disk — the local disk is wiped whenever the runtime disconnects. This cell mounts your Drive (it will ask for authorization once) and creates the output folders.

In [ ]:
OUTPUT_DIR = "results"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = DRIVE_OUTPUT_DIR
    except ImportError:
        print("Not running in Colab — saving locally instead.")

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Results will be written to: {OUTPUT_DIR}")


## 3. Download the dataset from Google Drive

In [ ]:
import gdown

os.makedirs(DATA_DIR, exist_ok=True)
gdown.download_folder(
    GDRIVE_FOLDER_URL,
    output=DATA_DIR,
    quiet=False,
    use_cookies=False,
)


## 4. Load the dataset

In [ ]:
import glob
import pandas as pd

def find_data_file(data_dir, explicit_path=None):
    if explicit_path:
        return explicit_path
    candidates = sorted(
        glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)
        + glob.glob(os.path.join(data_dir, "**", "*.xlsx"), recursive=True)
        + glob.glob(os.path.join(data_dir, "**", "*.xls"), recursive=True)
    )
    if not candidates:
        raise FileNotFoundError(
            f"No CSV/XLSX files found under '{data_dir}'. "
            "Set DATA_FILE explicitly to the path of your dataset."
        )
    return candidates[0]

data_path = find_data_file(DATA_DIR, DATA_FILE)
print(f"Loading: {data_path}")

if data_path.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(data_path)
else:
    # A few common finance-news CSVs (e.g. Financial PhraseBank exports) are
    # semicolon-delimited with no header and latin-1 encoded. Try the
    # standard format first and fall back if it looks malformed.
    try:
        df = pd.read_csv(data_path)
        if df.shape[1] == 1:
            raise ValueError("Only one column parsed, retrying with ';' delimiter")
    except (ValueError, UnicodeDecodeError):
        df = pd.read_csv(data_path, sep=";", header=None, encoding="latin-1",
                          names=["sentiment", "text"])

print(df.shape)
df.head()


## 5. Detect the text (and optional label) column

In [ ]:
TEXT_CANDIDATES = [
    "text", "Text", "news", "News", "headline", "Headline", "title", "Title",
    "sentence", "Sentence", "content", "Content", "article", "Article",
    "News Headline", "body", "Body",
]
LABEL_CANDIDATES = [
    "sentiment", "Sentiment", "label", "Label", "target", "Target", "class", "Class",
]

def resolve_column(df, configured, candidates, required=True):
    if configured not in (None, "auto"):
        if configured not in df.columns:
            raise KeyError(f"Configured column '{configured}' not found in {list(df.columns)}")
        return configured
    if configured is None:
        return None
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"Could not auto-detect a text column among {list(df.columns)}. "
            "Set TEXT_COLUMN explicitly in the config cell."
        )
    return None

text_col = resolve_column(df, TEXT_COLUMN, TEXT_CANDIDATES, required=True)
label_col = resolve_column(df, LABEL_COLUMN, LABEL_CANDIDATES, required=False)

print(f"Text column:  {text_col!r}")
print(f"Label column: {label_col!r}")

df = df.dropna(subset=[text_col]).reset_index(drop=True)
df[text_col] = df[text_col].astype(str)

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df = df.sample(n=SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Sampled {SAMPLE_N:,} of the rows (set SAMPLE_N = None for the full dataset)")

print(f"{len(df):,} rows to process")


## 6. Load the models

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

use_cuda = torch.cuda.is_available()
device = 0 if use_cuda else -1
# fp16 halves memory and roughly doubles throughput on GPU, with no
# measurable effect on which label wins
dtype = torch.float16 if use_cuda else torch.float32
print(f"Using device: {'cuda' if use_cuda else 'cpu'}, dtype: {dtype}")

MODELS = {
    "finbert": "ProsusAI/finbert",
    "roberta": "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "finbert_tone": "yiyanghkust/finbert-tone",
}

pipelines = {}
for key, model_name in MODELS.items():
    print(f"Loading {key} ({model_name})...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, torch_dtype=dtype)
    pipelines[key] = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        device=device,
        top_k=None,          # return scores for every class
        truncation=True,
        max_length=MAX_LENGTH,
    )
print("All models loaded.")


## 6b. Quick demo on toy examples

A tiny sanity check before committing to the full dataset: six hand-written headlines with obvious sentiment, classified by all three models side by side.

**You can run this without downloading any data** — only cells 1 (install), 2 (config), and 6 (load models) need to run first; the Drive download (sections 3–5) can be skipped.

Things to notice in the output:
- The two finance-tuned models usually call the rate-decision headline `neutral`, while the general-purpose RoBERTa often reads plain factual finance statements as negative or positive.
- The `_score` column is the winning class's probability — how confident each model is.

In [ ]:
toy_headlines = [
    "Company X shares surge 12% after record quarterly earnings beat expectations",
    "Regulators fine the bank $2 billion over money-laundering failures",
    "The central bank kept interest rates unchanged, as widely expected",
    "Tech giant announces layoffs of 10,000 employees amid slowing demand",
    "Oil prices edge higher on supply concerns; analysts remain cautious",
    "Startup files for bankruptcy after failing to secure new funding",
]

demo_rows = []
for text in toy_headlines:
    row = {"text": text}
    for key, pipe in pipelines.items():
        # pipe returns a list with one entry per input; that entry is a
        # list of {'label', 'score'} dicts, one per class
        scores = {d["label"].lower(): d["score"] for d in pipe(text)[0]}
        top = max(scores, key=scores.get)
        row[f"{key}_label"] = top
        row[f"{key}_score"] = round(scores[top], 3)
    demo_rows.append(row)

import pandas as pd
demo_df = pd.DataFrame(demo_rows)
demo_df


## 7. Run inference (chunked, resumable)

The data is processed in chunks of `CHUNK_SIZE` rows. Each finished chunk — original columns plus all three models' labels and per-class probabilities — is written to Drive as a Parquet shard before the next one starts.

**If Colab disconnects, just reconnect and rerun the notebook from the top**: chunks that already have a shard on Drive are skipped, so the run resumes exactly where it stopped. Rough throughput on a T4: ~2 GPU-hours per million rows for all three models, so the full ~7M dataset is a multi-session job — the checkpoints make that safe.

In [ ]:
import math
from tqdm.auto import tqdm

def classify_texts(pipe, texts):
    """Run one pipeline over a list of texts, returning a list of
    {label: probability} dicts. Batched manually so tqdm shows real progress."""
    results = []
    for start in tqdm(range(0, len(texts), BATCH_SIZE), leave=False):
        batch = texts[start : start + BATCH_SIZE]
        outs = pipe(batch, batch_size=BATCH_SIZE)
        results.extend({d["label"].lower(): d["score"] for d in out} for out in outs)
    return results

def attach_results(chunk, prefix, raw):
    labels = sorted({lbl for row in raw for lbl in row})
    for lbl in labels:
        chunk[f"{prefix}_{lbl}"] = [row.get(lbl, 0.0) for row in raw]
    top = [max(row, key=row.get) for row in raw]
    chunk[f"{prefix}_label"] = top
    chunk[f"{prefix}_score"] = [row[t] for row, t in zip(raw, top)]

n_chunks = math.ceil(len(df) / CHUNK_SIZE)
print(f"{len(df):,} rows -> {n_chunks} chunks of up to {CHUNK_SIZE:,}")

for i in range(n_chunks):
    shard_path = os.path.join(CHECKPOINT_DIR, f"shard_{i:05d}.parquet")
    if os.path.exists(shard_path):
        continue  # finished in a previous session — resume past it
    chunk = df.iloc[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE].copy()
    texts = chunk[text_col].tolist()
    for key, pipe in pipelines.items():
        attach_results(chunk, key, classify_texts(pipe, texts))
    chunk.to_parquet(shard_path, index=False)
    print(f"chunk {i + 1}/{n_chunks} saved -> {shard_path}")

print("All chunks done.")


## 8. Combine the shards

In [ ]:
shard_paths = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "shard_*.parquet")))
print(f"{len(shard_paths)} shards found")

result_df = pd.concat(
    [pd.read_parquet(p) for p in shard_paths], ignore_index=True
)
print(f"{len(result_df):,} rows, {len(result_df.columns)} columns")
result_df.head()


## 9. Save the combined results

Parquet is the primary output — at millions of rows it is several times smaller than CSV, loads back in seconds, and preserves column types exactly. A CSV copy is written too, but only when the result is small enough for that to be sensible (a full 7M-row CSV would be 4GB+).

In [ ]:
parquet_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_BASENAME}.parquet")
result_df.to_parquet(parquet_path, index=False)
print(f"Saved {len(result_df):,} rows -> {parquet_path}")

if len(result_df) <= 500_000:
    csv_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_BASENAME}.csv")
    result_df.to_csv(csv_path, index=False)
    print(f"Also saved CSV -> {csv_path}")
else:
    print("Skipping CSV copy (too many rows); load the Parquet with pd.read_parquet().")


## 10. Compare the models

### 10.1 Label distribution per model

In [ ]:
import matplotlib.pyplot as plt

label_cols = {key: f"{key}_label" for key in pipelines}

dist = pd.DataFrame({
    key: result_df[col].value_counts(normalize=True)
    for key, col in label_cols.items()
}).fillna(0).sort_index()

dist.plot(kind="bar", figsize=(8, 5), title="Predicted label distribution by model")
plt.ylabel("Share of articles")
plt.xlabel("Label")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

dist


### 10.2 Pairwise agreement between models

In [ ]:
from itertools import combinations
from sklearn.metrics import cohen_kappa_score

model_keys = list(pipelines.keys())
agreement_rows = []
for a, b in combinations(model_keys, 2):
    la, lb = result_df[label_cols[a]], result_df[label_cols[b]]
    agreement = (la == lb).mean()
    kappa = cohen_kappa_score(la, lb)
    agreement_rows.append({"model_a": a, "model_b": b, "agreement": agreement, "cohen_kappa": kappa})

agreement_df = pd.DataFrame(agreement_rows)
agreement_df


### 10.3 Accuracy vs. ground truth (if a label column was found)

In [ ]:
if label_col is not None:
    normalized_truth = result_df[label_col].astype(str).str.lower()
    acc_rows = []
    for key, col in label_cols.items():
        acc = (result_df[col] == normalized_truth).mean()
        acc_rows.append({"model": key, "accuracy": acc})
    display(pd.DataFrame(acc_rows))
else:
    print("No ground-truth label column detected/configured — skipping accuracy comparison.")


### 10.4 Examples where the three models disagree

In [ ]:
disagreements = result_df[
    (result_df[label_cols["finbert"]] != result_df[label_cols["roberta"]])
    | (result_df[label_cols["finbert"]] != result_df[label_cols["finbert_tone"]])
]

cols_to_show = [text_col] + list(label_cols.values())
disagreements[cols_to_show].head(20)


## Next steps

- Start with `SAMPLE_N = 100_000` to validate the pipeline end to end (~40 min on a T4), then set `SAMPLE_N = None` and rerun for the full dataset — already-finished shards are skipped, so nothing is recomputed.
- The results live on your Drive in `finsent_results/`: `sentiment_comparison_results.parquet` contains the original data plus, per model, `<model>_label`, `<model>_score`, and one `<model>_<class>` probability column per class. Load it with `pd.read_parquet(path)`; pass `columns=[...]` to read just the columns you need.
- Once a run is complete and saved, the `checkpoints/` shard folder can be deleted to free Drive space.
- Swap in any other Hugging Face model by adding an entry to the `MODELS` dict in section 6.